# 2.2 PySpark — Your Spark Knowledge, Python Syntax> ☕ You already know Spark from Java/Scala. This notebook maps your existing knowledge to PySpark.---

## Java Spark → PySpark Mapping| Java/Scala Spark | PySpark ||---|---|| `spark.read().parquet(path)` | `spark.read.parquet(path)` || `df.filter(col("age").gt(30))` | `df.filter(col("age") > 30)` || `df.groupBy("col").agg(count("*"))` | `df.groupBy("col").agg(count("*"))` || `df.withColumn("new", col("x").plus(1))` | `df.withColumn("new", col("x") + 1)` || `Dataset<Row>` | `DataFrame` (always untyped) |

In [ ]:
from pyspark.sql import SparkSessionfrom pyspark.sql import functions as Ffrom pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType# Create session — you know thisspark = SparkSession.builder \    .master("local[*]") \    .appName("HealthcareETL") \    .config("spark.sql.adaptive.enabled", "true") \    .getOrCreate()print(f"Spark version: {spark.version}")

In [ ]:
# Define schema explicitly — like you'd do in Javaschema = StructType([    StructField("patient_id", StringType(), False),    StructField("encounter_id", StringType(), False),    StructField("diagnosis_code", StringType(), True),    StructField("age", IntegerType(), True),    StructField("cost", DoubleType(), True),])# Create sample datadata = [    ("P001", "E001", "E11.9", 45, 1200.50),    ("P001", "E002", "I10",   45, 850.00),    ("P002", "E003", "E11.9", 62, 2100.75),    ("P002", "E004", "J44.1", 62, 3500.00),    ("P003", "E005", None,    38, 500.00),]df = spark.createDataFrame(data, schema)df.printSchema()df.show()

In [ ]:
# Transformations — identical logic to your Java Spark, just lighter syntaxresult = (    df    .filter(F.col("diagnosis_code").isNotNull())    .withColumn("cost_category",        F.when(F.col("cost") > 2000, "high")         .when(F.col("cost") > 1000, "medium")         .otherwise("low")    )    .groupBy("diagnosis_code")    .agg(        F.count("*").alias("encounter_count"),        F.round(F.avg("cost"), 2).alias("avg_cost"),        F.round(F.sum("cost"), 2).alias("total_cost"),    )    .orderBy(F.desc("total_cost")))result.show()

## UDFs — Python Functions in Spark☕ In Java you'd create a `UDF1<String, String>`. In Python it's just a decorator.

In [ ]:
from pyspark.sql.functions import udffrom pyspark.sql.types import StringType# Simple UDF@udf(returnType=StringType())def classify_icd(code: str) -> str:    """Map ICD-10 codes to categories."""    if code is None:        return "unknown"    prefix = code[0]    mapping = {"E": "endocrine", "I": "circulatory", "J": "respiratory"}    return mapping.get(prefix, "other")df_classified = df.withColumn("category", classify_icd(F.col("diagnosis_code")))df_classified.show()

In [ ]:
# 💡 Prefer Pandas UDFs for performance (vectorized, uses Arrow)from pyspark.sql.functions import pandas_udfimport pandas as pd@pandas_udf("double")def normalize_cost(costs: pd.Series) -> pd.Series:    """Z-score normalization — runs in batches via Apache Arrow."""    return (costs - costs.mean()) / costs.std()df.withColumn("cost_normalized", normalize_cost(F.col("cost"))).show()

In [ ]:
# Window functions — same as Java Sparkfrom pyspark.sql.window import Windowwindow_spec = Window.partitionBy("patient_id").orderBy(F.desc("cost"))df_ranked = (    df    .withColumn("cost_rank", F.row_number().over(window_spec))    .withColumn("patient_total", F.sum("cost").over(        Window.partitionBy("patient_id")    )))df_ranked.show()

In [ ]:
spark.stop()